# Import libraries

In [ ]:
import os
import subprocess
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 3D plotting toolkit
import statsmodels.api as sm
import scipy.stats as stats
import importlib.util
import sys

In [ ]:
from scripts.parse_vars import parse_variables

In [ ]:
path_vars = f"../../geno_simulation.txt"
R_directory = f"../../rstudio_geno_simulation"
R_file = f"create_geno.R"

variables = parse_variables(path_vars)
if 'G' not in globals():
    G = int(variables['G'])
if 'L' not in globals():
    L = int(variables['L'])
if 'c' not in globals():
    c = int(variables['c'])
if 'k' not in globals():
    k = int(variables['k'])
if 'M' not in globals():
    M = float(variables['M'])
if 'HWE' not in globals():
    HWE = int(variables['HWE'])

if 'tools' not in globals():
    tools = ['PCA', 'abyss_counted', 'abyss', 'no_corr']


if 'scenarios' not in globals():
    scenarios = ['snp_effect',
                 'linear_continuous',
                 'non_linear_continuous',
                 'discrete_global',
                 'discrete_localized',
                 'mix_linear_continuous',
                 'mix_non_linear_continuous',
                 'mix_discrete_global',
                 'mix_discrete_localized']

if 'very_rare_threshold_L' not in globals():
    very_rare_threshold_L = float(variables['very_rare_threshold_L'])
if 'very_rare_threshold_H' not in globals():
    very_rare_threshold_H = float(variables['very_rare_threshold_H'])
if 'rare_threshold_L' not in globals():
    rare_threshold_L = float(variables['rare_threshold_L'])
if 'rare_threshold_H' not in globals():
    rare_threshold_H = float(variables['rare_threshold_H'])
if 'common_threshold_L' not in globals():
    common_threshold_L = float(variables['common_threshold_L'])
if 'common_threshold_H' not in globals():
    common_threshold_H = float(variables['common_threshold_H'])
if 'F' not in globals():
    F = float(variables['F'])

# Load variables

In [ ]:
path_geno = f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/genotype"
geno = pd.read_pickle((f"{path_geno}/complete_inbred.pkl"))

In [ ]:
path_pheno = f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/phenotype/"
pheno = pd.read_pickle(f"{path_pheno}/env_pheno.pkl")
humans = pd.read_pickle(f"{path_pheno}/inbred_humans.pkl")

In [ ]:
pheno

In [ ]:
pheno_list = list(pheno[[col for col in pheno.columns if col not in ['x', 'y']]].columns)

In [ ]:
pheno_list

In [ ]:
nr_pcs = 5

In [ ]:
geno_pcs = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/PCs/geno_{nr_pcs}_PCs.pkl")
min_pcs = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/PCs/minor_{nr_pcs}_PCs.pkl")
het_pcs = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/PCs/het_{nr_pcs}_PCs.pkl")
maj_pcs = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/PCs/major_{nr_pcs}_PCs.pkl")


In [ ]:
geno_reconstruct = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/geno_reconstruct_{nr_pcs}_PCs.pkl")
min_reconstruct = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/minor_reconstruct_{nr_pcs}_PCs.pkl")
het_reconstruct = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/het_reconstruct_{nr_pcs}_PCs.pkl")
maj_reconstruct = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/major_reconstruct_{nr_pcs}_PCs.pkl")

In [ ]:
error = 1-(min_reconstruct+ het_reconstruct+ maj_reconstruct)

In [ ]:
row_means = (geno_reconstruct - ((maj_reconstruct - min_reconstruct) + error)).mean(axis=0)
#row_means = (geno_reconstruct - (((1-min_reconstruct-het_reconstruct) - min_reconstruct))).mean(axis=0)



In [ ]:
row_means

# PCA

In [ ]:
geno_pcs

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(geno_pcs['PC1'], geno_pcs['PC2'], s=5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Genotype Matrix")
plt.show()


# Gwas

In [ ]:
env_factor = "04_pheno_sine_x_y"

In [ ]:
from statsmodels.api import OLS, add_constant
import numpy as np
import pandas as pd

# Extract phenotype
y = pheno[env_factor].values

# Initialize results list
results = []

# Loop through SNPs
for snp in geno.columns:
    x = geno[snp].values
    # Combine SNP with PCs
    X = pd.concat([pd.Series(x, name='SNP'), geno_pcs], axis=1)
    X = add_constant(X)  # Add intercept
    model = OLS(y, X).fit()
    pval = model.pvalues['SNP']  # p-value for SNP term
    logp = -np.log10(pval)
    results.append((snp, logp))

# Create and sort results DataFrame
gwas_results = pd.DataFrame(results, columns=['SNP', '-log10(p)'])
gwas_results = gwas_results.sort_values('-log10(p)', ascending=False).reset_index(drop=True)

# Display
gwas_results


In [ ]:


plt.figure(figsize=(12, 6))
plt.plot(gwas_results['-log10(p)'].values, marker='o', linestyle='none')
plt.axhline(-np.log10(5e-8), color='red', linestyle='--', label='Genome-wide significance')
plt.xlabel('SNPs')
plt.ylabel('-log10(p-value)')
plt.title(f'GWAS Manhattan-like Plot for {env_factor}')
plt.legend()
plt.show()

In [ ]:


# Extract MAF from SNP names
gwas_results['MAF'] = gwas_results['SNP'].str.extract(r'MAF_([\d.]+)').astype(float)

plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(
    data=gwas_results,
    x="MAF",
    y="-log10(p)",
    hue="MAF",
    palette="viridis",
    s=10,
    alpha=0.7,
    edgecolor=None
)


In [ ]:
gwas_results['maf_deviation'] = gwas_results['SNP'].map(row_means)
gwas_results['maf_deviation_abs'] = gwas_results['maf_deviation'].abs()

In [ ]:
gwas_results

In [ ]:

plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(
    data=gwas_results,
    x="maf_deviation",
    y="-log10(p)",
    hue="MAF",
    palette="viridis",
    s=10,
    alpha=0.7,
    edgecolor=None
)

# Lets iterate through the PCs 

In [ ]:
nrs_pcs = list(set([int(f.split("_")[1]) for f in os.listdir(f"simulation_data//G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/PCs")]))

In [ ]:
done_pheno_list = [f.split(".pkl")[0] for f in os.listdir(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/comparaisons") if f.endswith(".pkl")]

In [ ]:
pheno_list_to_do = [p for p in pheno_list if p not in done_pheno_list]

In [ ]:
"""
os.makedirs(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/comparaisons", exist_ok=True)

EPS = np.finfo(float).tiny  # smallest positive float

for env_factor in pheno_list_to_do:
    print(env_factor)
    dfs = []

    for nr_pcs in nrs_pcs:
        geno_pcs = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/PCs/geno_{nr_pcs}_PCs.pkl")
        geno_reconstruct = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/geno_reconstruct_{nr_pcs}_PCs.pkl")
        min_reconstruct  = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/minor_reconstruct_{nr_pcs}_PCs.pkl")
        het_reconstruct  = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/het_reconstruct_{nr_pcs}_PCs.pkl")
        maj_reconstruct  = pd.read_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/maf_reconstruct/major_reconstruct_{nr_pcs}_PCs.pkl")

        error = 1 - (min_reconstruct + het_reconstruct + maj_reconstruct)
        row_means = (geno_reconstruct - ((maj_reconstruct - min_reconstruct) + error)).mean(axis=0)

        y = pheno[env_factor].values

        results = []
        for snp in geno.columns:   # ensure 'geno' is defined/loaded; otherwise use geno_reconstruct.columns
            x = geno[snp].values
            X = pd.concat([pd.Series(x, name='SNP'), geno_pcs], axis=1)
            X = add_constant(X, has_constant='add')
            model = OLS(y, X).fit()

            pval = model.pvalues.get('SNP', np.nan)
            if np.isnan(pval):
                logp = 0.0
            else:
                logp = -np.log10(np.clip(pval, EPS, 1.0))

            results.append((snp, logp))

        gwas_results = pd.DataFrame(results, columns=['SNP', '-log10(p)'])
        gwas_results = gwas_results.sort_values('-log10(p)', ascending=False, kind='mergesort').reset_index(drop=True)

        gwas_results['MAF'] = gwas_results['SNP'].str.extract(r'MAF_([\d.]+)').astype(float)
        gwas_results['nr_PCs'] = nr_pcs
        gwas_results['maf_deviation'] = gwas_results['SNP'].map(row_means)

        dfs.append(gwas_results)

    # Combine all PC settings for this phenotype
    df = pd.concat(dfs, axis=0, ignore_index=True)
    df.to_pickle(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/comparaisons/{env_factor}.pkl")
"""

In [ ]:
comparaison_files = [f for f in os.listdir(f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/comparaisons/") if f.startswith("0")]

In [ ]:
comparaison_files

In [ ]:
dfs = []
for nr_index in range(len(comparaison_files)):
    path_file = f"simulation_data/G{G}_L{L}_c{c}_k{k}_M{M}_F{F}/comparaisons/{comparaison_files[nr_index]}"
    df = pd.read_pickle(path_file)
    df['env'] = comparaison_files[nr_index].split(".pkl")[0]
    dfs.append(df)

In [ ]:
df = pd.concat(dfs, axis=0, ignore_index=True)


In [ ]:
df

In [ ]:

plt.figure(figsize=(10,6))
for pcs in df['nr_PCs'].unique():
    subset = df[df['nr_PCs'] == pcs]
    plt.scatter(subset['MAF'], subset['maf_deviation'], s=5, alpha=0.5, label=f"{pcs} PCs")

plt.axhline(0, color="black", linestyle="--")
plt.xlabel("MAF")
plt.ylabel("MAF deviation")
plt.legend()
plt.title("Reconstructed MAF deviation vs. true MAF by number of PCs")
plt.show()


In [ ]:
fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111, projection='3d')

# Scatter all points in 3D
sc = ax.scatter(
    df['MAF'], 
    df['maf_deviation'], 
    df['nr_PCs'], 
    c=df['-log10(p)'],   # color by -log10(p) if you want
    cmap="viridis", 
    alpha=0.6, s=10
)

ax.set_xlabel("MAF")
ax.set_ylabel("MAF deviation")
ax.set_zlabel("Number of PCs")
ax.set_title("3D view: MAF vs deviation vs PCs")

# Add colorbar for -log10(p)
cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label("-log10(p)")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="nr_PCs", y="maf_deviation")
plt.axhline(0, color="black", linestyle="--")
plt.title("Distribution of MAF deviation across number of PCs")
plt.show()


In [ ]:
g = sns.FacetGrid(df, col="env", col_wrap=3, height=4, sharey=False)
g.map_dataframe(sns.scatterplot, x="MAF", y="maf_deviation", hue="nr_PCs", alpha=0.5, s=5)
g.add_legend()
g.set_axis_labels("MAF", "MAF deviation")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

# Bin MAF into categories
df['MAF_bin'] = pd.cut(df['MAF'], bins=np.linspace(0,0.5,11))

pivot = df.groupby(['MAF_bin','nr_PCs'])['maf_deviation'].mean().unstack()

plt.figure(figsize=(10,6))
sns.heatmap(pivot, cmap="coolwarm", center=0, annot=False)
plt.title("Mean MAF deviation by allele frequency and number of PCs")
plt.ylabel("MAF bin")
plt.xlabel("nr_PCs")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.scatterplot(
    data=df, x="maf_deviation", y="-log10(p)", 
    hue="nr_PCs", alpha=0.5, s=5
)
plt.xlabel("MAF deviation")
plt.ylabel("-log10(p)")
plt.title("Effect of MAF deviation on association strength")
plt.show()


In [ ]:
corrs = df.groupby("nr_PCs")[["maf_deviation","-log10(p)"]].corr().iloc[0::2, -1]
print(corrs)


In [ ]:
g = sns.FacetGrid(df, col="env", col_wrap=3, height=4, sharey=False)
g.map_dataframe(
    sns.scatterplot, x="maf_deviation", y="-log10(p)", hue="nr_PCs", alpha=0.5, s=5
)
g.add_legend()
plt.show()


In [ ]:
sns.lmplot(
    data=df, x="maf_deviation", y="-log10(p)",
    hue="nr_PCs", col="env", col_wrap=3,
    scatter_kws={"alpha":0.2, "s":5}, lowess=True
)

In [ ]:
df['MAF_bin'] = pd.cut(df['MAF'], bins=[0,0.05,0.1,0.2,0.3,0.4,0.5])

plt.figure(figsize=(12,6))
sns.boxplot(data=df, x="MAF_bin", y="-log10(p)", hue="nr_PCs")
plt.xticks(rotation=45)
plt.title("-log10(p) across MAF bins, stratified by number of PCs")
plt.show()


In [ ]:
# Optional: make it look like seaborn
sns.set(style="whitegrid")

# Extract variables
x = df['nr_PCs']
y = df['maf_deviation']
z = df['-log10(p)']
hue = df['MAF']

# Create figure
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Scatter with hue as MAF
sc = ax.scatter(x, y, z,
                c=hue,
                cmap='coolwarm',
                s=50,
                alpha=0.8)

# Labels
ax.set_xlabel('Number of PCs')
ax.set_ylabel('MAF deviation')
ax.set_zlabel('-log10(p)')

# Color bar for MAF
cbar = fig.colorbar(sc, ax=ax, shrink=0.5, aspect=10)
cbar.set_label('MAF')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Assuming df is your dataframe
bell_curves = []

for pcs, group in df.groupby("nr_PCs"):
    mean_dev = group["maf_deviation"].mean()
    std_dev = group["maf_deviation"].std(ddof=0)  # population std
    
    # Convert to z-scores
    z_scores = (group["maf_deviation"] - mean_dev) / std_dev
    
    bell_curves.append(pd.DataFrame({
        "nr_PCs": pcs,
        "maf_deviation": group["maf_deviation"],
        "z_score": z_scores
    }))

bell_curve_df = pd.concat(bell_curves)


In [ ]:
bell_curve_df

In [ ]:
import matplotlib.pyplot as plt

for pcs in bell_curve_df["nr_PCs"].unique():
    subset = bell_curve_df[bell_curve_df["nr_PCs"] == pcs]
    plt.hist(subset["maf_deviation"], bins=30, density=True, alpha=0.5, label=f"PCs={pcs}")

plt.legend()
plt.title("MAF deviation distributions by nr_PCs (z-scores)")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Assign numeric positions for nr_PCs so we can place them on the z-axis
pcs_positions = {pcs: i for i, pcs in enumerate(sorted(bell_curve_df["nr_PCs"].unique()))}

for pcs in bell_curve_df["nr_PCs"].unique():
    subset = bell_curve_df[bell_curve_df["nr_PCs"] == pcs]
    
    # Histogram data
    hist_vals, bin_edges = np.histogram(subset["z_score"], bins=30, density=True)
    bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
    
    # Z position for all bars in this group
    z_pos = np.full_like(bin_centers, pcs_positions[pcs], dtype=float)
    
    # Width and depth for bars
    dx = (bin_edges[1] - bin_edges[0]) * 0.9
    dy = 0.5  # thickness for the PC axis
    
    # Plot bars
    ax.bar3d(bin_centers, z_pos, np.zeros_like(hist_vals),
             dx, dy, hist_vals, alpha=0.6)

# Labels
ax.set_xlabel("Z-score (MAF deviation)")
ax.set_ylabel("nr_PCs")
ax.set_zlabel("Density")
ax.set_yticks(list(pcs_positions.values()))
ax.set_yticklabels(list(pcs_positions.keys()))
ax.set_title("MAF deviation distributions by nr_PCs (3D view)")

plt.show()
